<a href="https://colab.research.google.com/github/Eikthyrnir/CatVTON_3DGS/blob/main/MV_VTON_with_MVG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MV-VTON Inference with MVG Dataset

This notebook runs MV-VTON inference using the official MVG dataset on Google Colab (T4 GPU).

**Prerequisites:**
- MVG dataset uploaded to Google Drive
- Colab GPU runtime (T4 or better, ~15GB VRAM)

**Pipeline:**
1. Install dependencies
2. Clone repo & apply compatibility patches
3. Download checkpoints (mvg.ckpt + vgg19)
4. Mount Drive & assemble MVG dataset
5. Run inference
6. View results

## Step 1: Check GPU & Install Dependencies

In [1]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

Sun May 24 17:15:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q pytorch-lightning==1.9.5 \
    "transformers>=4.27,<4.40" \
    "diffusers==0.21.4" \
    "einops>=0.3" \
    omegaconf==2.3.0 \
    "kornia>=0.6,<0.8" \
    "albumentations>=1.0,<2.0" \
    "huggingface-hub==0.19.4" \
    safetensors==0.4.1 \
    scipy==1.11.4 \
    scikit-image==0.20.0 \
    "imageio>=2.9" \
    "torchmetrics>=0.6,<1.0" \
    taming-transformers-rom1504 \
    pyyaml tqdm

# Install CLIP
!pip install -q git+https://github.com/openai/CLIP.git

print("\nDependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 109.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 152.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: Clone Repo & Apply Patches

In [3]:
import os
if not os.path.exists('/content/MV-VTON'):
    !git clone https://github.com/hywang2002/MV-VTON.git /content/MV-VTON
else:
    print('Repo already cloned.')

# Copy paired dataset as default
!cp /content/MV-VTON/ldm/data/cp_dataset_mv_paired.py /content/MV-VTON/ldm/data/cp_dataset.py
print('Done.')

Cloning into '/content/MV-VTON'...
remote: Enumerating objects: 307, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 307 (delta 23), reused 21 (delta 21), pack-reused 282 (from 1)
Receiving objects: 100% (307/307), 1.31 MiB | 33.44 MiB/s, done.
Resolving deltas: 100% (123/123), done.
Done.


In [4]:
with open('/content/MV-VTON/ldm/models/local_controlnet.py', 'r') as f:
    content = f.read()
# Revert any previous broken patch, then leave original imports intact (diffusers==0.21.4 is compatible)
content = content.replace(
    'from diffusers.loaders import FromOriginalModelMixin as FromOriginalControlnetMixin',
    'from diffusers.loaders import FromOriginalControlnetMixin'
)
content = content.replace(
    'from diffusers.models.unets.unet_2d_blocks import',
    'from diffusers.models.unet_2d_blocks import'
)
content = content.replace(
    'from diffusers.models.unets.unet_2d_condition import',
    'from diffusers.models.unet_2d_condition import'
)
with open('/content/MV-VTON/ldm/models/local_controlnet.py', 'w') as f:
    f.write(content)
# --- Patch ddpm.py: fix hardcoded Paint-by-Example path ---
with open('/content/MV-VTON/ldm/models/diffusion/ddpm.py', 'r') as f:
    content = f.read()
content = content.replace(
    '/mnt/pfs-mc0p4k/cvg/team/didonglin/why/my_huggingface/Paint-by-Example',
    'Fantasy-Studio/Paint-by-Example'
)
# Fix pytorch_lightning.utilities.distributed import (deprecated in 1.8+)
content = content.replace(
    'from pytorch_lightning.utilities.distributed import rank_zero_only',
    'from pytorch_lightning.utilities import rank_zero_only'
)
# Fix torch.load calls to add weights_only=False
content = content.replace(
    'torch.load("models/vgg/vgg19_conv.pth", map_location="cpu")',
    'torch.load("models/vgg/vgg19_conv.pth", map_location="cpu", weights_only=False)'
)
with open('/content/MV-VTON/ldm/models/diffusion/ddpm.py', 'w') as f:
    f.write(content)
# --- Patch test.py: torch.load weights_only ---
with open('/content/MV-VTON/test.py', 'r') as f:
    content = f.read()
content = content.replace(
    'torch.load(ckpt, map_location="cpu")',
    'torch.load(ckpt, map_location="cpu", weights_only=False)'
)
with open('/content/MV-VTON/test.py', 'w') as f:
    f.write(content)
print('All patches applied successfully.')

All patches applied successfully.


In [5]:
# import os
# unet_path = None
# for root, dirs, files in os.walk("/content/hf_cache"):
#     if "unet" in dirs:
#         candidate = os.path.join(root, "unet")
#         if os.path.exists(os.path.join(candidate, "diffusion_pytorch_model.bin")):
#             unet_path = candidate
#             break
#     # Also check for config.json that indicates UNet
#     if "diffusion_pytorch_model.bin" in files and "config.json" in files:
#         with open(os.path.join(root, "config.json")) as f:
#             if "UNet2DConditionModel" in f.read():
#                 unet_path = root
#                 break
# print(f"UNet path: {unet_path}")
# # Patch ddpm.py to use the direct local path instead of HF repo name
# with open('/content/MV-VTON/ldm/models/diffusion/ddpm.py', 'r') as f:
#     content = f.read()
# # Replace the from_pretrained call to use local path without subfolder
# old = '''UNet2DConditionModel.from_pretrained(
#             "Fantasy-Studio/Paint-by-Example", subfolder="unet")'''
# new = f'''UNet2DConditionModel.from_pretrained("{unet_path}")'''
# if old not in content:
#     # Try original path
#     old = '''UNet2DConditionModel.from_pretrained(
#             "/mnt/pfs-mc0p4k/cvg/team/didonglin/why/my_huggingface/Paint-by-Example", subfolder="unet")'''
#     new = f'''UNet2DConditionModel.from_pretrained("{unet_path}")'''
# content = content.replace(old, new)
# with open('/content/MV-VTON/ldm/models/diffusion/ddpm.py', 'w') as f:
#     f.write(content)
# print("Patched to use local UNet path.")

## Step 3: Download Checkpoints

In [6]:
import os

# Download mvg.ckpt
ckpt_path = '/content/MV-VTON/checkpoints/mvg.ckpt'
os.makedirs('/content/MV-VTON/checkpoints', exist_ok=True)

if not os.path.exists(ckpt_path) and not os.path.exists('/content/MV-VTON/checkpoints/extracted_mvg'):
    !gdown --fuzzy "https://drive.google.com/file/d/1J91PoT8A9yqHWNxkgRe6ZCnDEhN-H9O6/view" -O {ckpt_path}
else:
    print('Checkpoint already exists.')

# Download vgg19_conv.pth
vgg_path = '/content/MV-VTON/models/vgg/vgg19_conv.pth'
os.makedirs('/content/MV-VTON/models/vgg', exist_ok=True)

if not os.path.exists(vgg_path):
    !gdown --fuzzy "https://drive.google.com/file/d/1rvow8jStPt8t2prDcSRlnf8yzXhrYeGo/view" -O {vgg_path}
else:
    print('VGG checkpoint already exists.')

Downloading...
From (original): https://drive.google.com/uc?id=1J91PoT8A9yqHWNxkgRe6ZCnDEhN-H9O6
From (redirected): https://drive.google.com/uc?id=1J91PoT8A9yqHWNxkgRe6ZCnDEhN-H9O6&confirm=t&uuid=02af43db-36e6-4dca-b68c-c1daf6c0bb90
To: /content/MV-VTON/checkpoints/mvg.ckpt
100% 7.26G/7.26G [01:48<00:00, 66.7MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1rvow8jStPt8t2prDcSRlnf8yzXhrYeGo
From (redirected): https://drive.google.com/uc?id=1rvow8jStPt8t2prDcSRlnf8yzXhrYeGo&confirm=t&uuid=1c59f715-46a0-4b5c-975f-522826539863
To: /content/MV-VTON/models/vgg/vgg19_conv.pth
100% 160M/160M [00:04<00:00, 35.8MB/s]


In [7]:
# Decompress & extract checkpoint if needed
import os, tarfile, gzip

ckpt_path = '/content/MV-VTON/checkpoints/mvg.ckpt'
extract_dir = '/content/MV-VTON/checkpoints/extracted_mvg'

if os.path.exists(extract_dir):
    print('Already extracted.')
    !find {extract_dir} -name '*.ckpt'
else:
    size = os.path.getsize(ckpt_path)
    print(f'Checkpoint size: {size / 1024**2:.1f} MB')

    with open(ckpt_path, 'rb') as f:
        header = f.read(4)

    # Decompress gzip if needed
    if header[0] == 0x1f and header[1] == 0x8b:
        print('File is gzip compressed. Decompressing...')
        !mv {ckpt_path} {ckpt_path}.gz
        !gunzip {ckpt_path}.gz
        print(f'Decompressed. New size: {os.path.getsize(ckpt_path) / 1024**2:.1f} MB')

    # Extract tar if needed
    if tarfile.is_tarfile(ckpt_path):
        print('File is a tar archive. Extracting...')
        with tarfile.open(ckpt_path) as tar:
            tar.extractall(extract_dir)
        !find {extract_dir} -name '*.ckpt'
    else:
        print('File is a direct PyTorch checkpoint (no extraction needed).')
        os.makedirs(extract_dir, exist_ok=True)
        os.symlink(ckpt_path, os.path.join(extract_dir, 'mvg.ckpt'))

Checkpoint size: 6920.1 MB
File is gzip compressed. Decompressing...
Decompressed. New size: 7986.5 MB
File is a tar archive. Extracting...


/tmp/ipykernel_11028/4191551371.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


/content/MV-VTON/checkpoints/extracted_mvg/ckpt_mv/epoch=000006.ckpt


## Step 4: Mount Drive & Assemble MVG Dataset

The MVG data is spread across multiple Drive folders:
- `image_ori/` → person images without background
- `MVG/test/cloth/` & `cloth-mask/` → clothing images
- `MVG-pre-warped/test/warp_feat/` → pre-warped clothing
- `MVG/test/image-densepose/` → used as skeleton proxy
- `MVG/test/image-parse/` + `agnostic-wo-bg/` → used to generate inpaint masks

In [9]:
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURE PATHS ===
DRIVE_ROOT = '/content/drive/MyDrive/magisterka/mvg'
DATA_ROOT = '/content/mvvton_data'
TEST_PATH = f'{DATA_ROOT}/test'

Mounted at /content/drive


In [12]:
!mkdir -p /content/datasets/MVG
!cp -r /content/drive/MyDrive/weavella/datasets/MVG /content/datasets/

In [13]:
!unzip -q /content/datasets/MVG/MVG.zip -d /content/datasets/MVG/
!unzip -q /content/datasets/MVG/MVG-pre-warped.zip -d /content/datasets/MVG/

In [ ]:
# # First, ensure gdown is up to date
# !pip install -U -q gdown

# # Replace this URL with your actual Google Drive folder share link
# folder_url = 'https://drive.google.com/drive/folders/YOUR_FOLDER_ID_HERE'

# # Download the folder
# !gdown --folder {folder_url} -O /content/downloaded_folder

In [14]:
!cp /content/drive/MyDrive/weavella/images/Photos_Manual_Selection/9/vadim.MOV /content/datasets
!cp -r /content/drive/MyDrive/weavella/images/Photos_Manual_Selection/9/no_bg /content/datasets

In [16]:
!mkdir -p /content/datasets/init_frames
!ffmpeg -i /content/datasets/vadim.MOV -qscale: 1 -qmin 1 -vf fps=2 /content/datasets/init_frames/%04d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [15]:
import os
import shutil

flat_dir = '/content/datasets/no_bg'
structured_dir = '/content/datasets/no_bg_structured'
os.makedirs(structured_dir, exist_ok=True)

for file in os.listdir(flat_dir):
    if file.lower().endswith(('.jpg', '.png', '.jpeg')):
        name = os.path.splitext(file)[0]
        item_dir = os.path.join(structured_dir, name)
        os.makedirs(item_dir, exist_ok=True)

        # MV-VTON expects 3 views per person (front=1, person=2, back=3)
        # Duplicate the image to prevent dataloader errors
        src_path = os.path.join(flat_dir, file)
        for i in range(1, 4):
            shutil.copy(src_path, os.path.join(item_dir, f'{i}.jpg'))

print(f"Restructured {len(os.listdir(structured_dir))} images into MV-VTON multi-view format.")

Restructured 58 images into MV-VTON multi-view format.


In [16]:
DRIVE_ROOT = '/content/datasets/MVG'
DATA_ROOT = '/content/mvvton_data'
TEST_PATH = f'{DATA_ROOT}/test'

In [17]:
import os
import shutil

os.makedirs(TEST_PATH, exist_ok=True)

# Symlink existing directories
links = {
    'image-wo-bg': f'/content/datasets/no_bg_structured',
    'cloth': f'{DRIVE_ROOT}/MVG/test/cloth',
    'cloth-mask': f'{DRIVE_ROOT}/MVG/test/cloth-mask',
    'warp_feat': f'{DRIVE_ROOT}/MVG-pre-warped/test/warp_feat',
    'skeletons': f'{DRIVE_ROOT}/MVG/test/image-densepose',
}

for name, src in links.items():
    dst = os.path.join(TEST_PATH, name)

    # Remove old symlink or directory to force refresh
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)

    if os.path.exists(src):
        os.symlink(src, dst)
        count = len([x for x in os.listdir(dst) if not x.startswith('.')])
        print(f'  {name}: linked ({count} items)')
    else:
        print(f'  {name}: SOURCE NOT FOUND at {src}')

print('\nSymlinks created/updated.')

  image-wo-bg: linked (58 items)
  cloth: linked (591 items)
  cloth-mask: linked (591 items)
  warp_feat: linked (591 items)
  skeletons: linked (591 items)

Symlinks created/updated.


In [18]:
# Generate inpaint_mask
# First from image-parse (143 folders), then fill remaining from agnostic-wo-bg diff
import numpy as np
from PIL import Image
from tqdm import tqdm
import os
parse_root = f'{DRIVE_ROOT}/MVG/test/image-parse'
agnostic_root = f'{DRIVE_ROOT}/MVG/test/agnostic-wo-bg'
img_root = os.path.join(TEST_PATH, 'image-wo-bg')
mask_root = os.path.join(TEST_PATH, 'inpaint_mask')
os.makedirs(mask_root, exist_ok=True)
folders = sorted([f for f in os.listdir(img_root) if not f.startswith('.')])
generated = 0
# Upper-clothes label in ATR parsing format
UPPER_BODY_LABELS = {4}
for folder in tqdm(folders, desc='Generating inpaint_mask'):
    dst_dir = os.path.join(mask_root, folder)
    if os.path.exists(dst_dir) and len(os.listdir(dst_dir)) > 0:
        continue
    os.makedirs(dst_dir, exist_ok=True)
    src_dir = os.path.join(img_root, folder)
    parse_dir = os.path.join(parse_root, folder)
    agn_dir = os.path.join(agnostic_root, folder)
    for img_name in sorted(os.listdir(src_dir)):
        src_path = os.path.join(src_dir, img_name)
        if not os.path.isfile(src_path):
            continue  # skip broken symlinks or missing files
        out_name = os.path.splitext(img_name)[0] + '.png'
        # Method 1: Use image-parse if available
        parse_path = os.path.join(parse_dir, out_name)
        if not os.path.exists(parse_path):
            parse_path = os.path.join(parse_dir, img_name)
        if os.path.exists(parse_path):
            parse_img = np.array(Image.open(parse_path))
            mask = np.zeros(parse_img.shape[:2], dtype=np.uint8)
            for label in UPPER_BODY_LABELS:
                mask[parse_img == label] = 255
        # Method 2: Diff between original and agnostic
        elif os.path.exists(os.path.join(agn_dir, img_name)):
            img = np.array(Image.open(src_path).convert('RGB'))
            agn = np.array(Image.open(os.path.join(agn_dir, img_name)).convert('RGB'))
            diff = np.abs(img.astype(float) - agn.astype(float)).mean(axis=2)
            mask = (diff > 30).astype(np.uint8) * 255
        else:
            # Fallback: empty mask (512x384)
            mask = np.zeros((512, 384), dtype=np.uint8)
        Image.fromarray(mask).save(os.path.join(dst_dir, out_name))
    generated += 1
print(f'Generated inpaint_mask for {generated} folders')
print(f'Total: {len([x for x in os.listdir(mask_root) if not x.startswith(".")])} folders')

Generating inpaint_mask: 100%|██████████| 58/58 [00:00<00:00, 185.74it/s]

Generated inpaint_mask for 58 folders
Total: 58 folders


In [19]:
# Verify final dataset structure
import os

required_dirs = ['image-wo-bg', 'cloth', 'cloth-mask', 'warp_feat', 'inpaint_mask', 'skeletons']
print(f'Dataset at: {TEST_PATH}')
all_ok = True
for d in required_dirs:
    path = os.path.join(TEST_PATH, d)
    exists = os.path.exists(path)
    count = len([x for x in os.listdir(path) if not x.startswith('.')]) if exists else 0
    status = f'OK ({count} items)' if exists else 'MISSING'
    print(f'  {d}: {status}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nDataset ready!')
else:
    print('\nWARNING: Some directories still missing.')

Dataset at: /content/mvvton_data/test
  image-wo-bg: OK (58 items)
  cloth: OK (591 items)
  cloth-mask: OK (591 items)
  warp_feat: OK (591 items)
  inpaint_mask: OK (58 items)
  skeletons: OK (591 items)

Dataset ready!


## Step 5: Run Inference

**Note:** First run will download Paint-by-Example UNet from HuggingFace (~2GB).
The model will appear to hang after "Running in eps-prediction mode" — this is normal, wait 2-5 min.

In [ ]:
from huggingface_hub import snapshot_download
import os
snapshot_download("Fantasy-Studio/Paint-by-Example", cache_dir="/content/hf_cache")
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"
print("Paint-by-Example downloaded.")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

model_index.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/524 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

model.ckpt:   0%|          | 0.00/12.6G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

diffusion_pytorch_model.bin:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

diffusion_pytorch_model.bin:   0%|          | 0.00/335M [00:00<?, ?B/s]

In [ ]:
!free -h
!du -sh /content/hf_cache/ 2>/dev/null
!du -sh /content/MV-VTON/checkpoints/ 2>/dev/null

In [ ]:
import os, glob
extract_dir = '/content/MV-VTON/checkpoints/extracted_mvg'
ckpts = glob.glob(os.path.join(extract_dir, '**/*.ckpt'), recursive=True)
if ckpts:
    CKPT_PATH = ckpts[0]
else:
    CKPT_PATH = '/content/MV-VTON/checkpoints/mvg.ckpt'
DATA_ROOT = '/content/mvvton_data'
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"
print(f'Using checkpoint: {CKPT_PATH}')
print(f'Using dataset: {DATA_ROOT}')
%cd /content/MV-VTON
!HF_HOME=/content/hf_cache python test.py --gpu_id 0 \
    --ddim_steps 10 \
    --outdir results/output/ \
    --config configs/viton512.yaml \
    --dataroot {DATA_ROOT} \
    --ckpt {CKPT_PATH} \
    --n_samples 1 \
    --seed 23 \
    --scale 1 \
    --H 128 --W 96 2>&1

Using checkpoint: /content/MV-VTON/checkpoints/extracted_mvg/ckpt_mv/epoch=000006.ckpt
Using dataset: /content/mvvton_data
/content/MV-VTON
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Global seed set to 23
Loading model from /content/MV-VTON/checkpoints/extracted_mvg/ckpt_mv/epoch=000006.ckpt
Global Step: 2135
LatentTryOnDiffusion: Running in eps-prediction mode


In [ ]:
# import os, glob

# # Find the actual checkpoint path
# extract_dir = '/content/MV-VTON/checkpoints/extracted_mvg'
# ckpts = glob.glob(os.path.join(extract_dir, '**/*.ckpt'), recursive=True)
# if ckpts:
#     CKPT_PATH = ckpts[0]
# else:
#     CKPT_PATH = '/content/MV-VTON/checkpoints/mvg.ckpt'

# DATA_ROOT = '/content/mvvton_data'

# print(f'Using checkpoint: {CKPT_PATH}')
# print(f'Using dataset: {DATA_ROOT}')

# %cd /content/MV-VTON

# !python test.py --gpu_id 0 \
#     --ddim_steps 50 \
#     --outdir results/output/ \
#     --config configs/viton512.yaml \
#     --dataroot {DATA_ROOT} \
#     --ckpt {CKPT_PATH} \
#     --n_samples 1 \
#     --seed 23 \
#     --scale 1 \
#     --H 512 --W 384

Using checkpoint: /content/MV-VTON/checkpoints/extracted_mvg/ckpt_mv/epoch=000006.ckpt
Using dataset: /content/mvvton_data
/content/MV-VTON
Global seed set to 23
Loading model from /content/MV-VTON/checkpoints/extracted_mvg/ckpt_mv/epoch=000006.ckpt
Global Step: 2135
LatentTryOnDiffusion: Running in eps-prediction mode


## Step 6: View Results

In [ ]:
import glob
import os
from PIL import Image
import matplotlib.pyplot as plt

result_dir = '/content/MV-VTON/results/output/upper_body'
results = sorted(glob.glob(os.path.join(result_dir, '*.png')))
print(f'Generated {len(results)} results')

# Show first 6 results
if results:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for i, ax in enumerate(axes.flat):
        if i < len(results):
            img = Image.open(results[i])
            ax.imshow(img)
            ax.set_title(os.path.basename(results[i]))
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results found. Check inference output for errors.')

Generated 0 results
No results found. Check inference output for errors.


In [ ]:
# Save results to Drive
import shutil

output_drive = '/content/drive/MyDrive/MV-VTON_results'
os.makedirs(output_drive, exist_ok=True)

result_dir = '/content/MV-VTON/results/output/upper_body'
if os.path.exists(result_dir):
    for f in glob.glob(os.path.join(result_dir, '*.png')):
        shutil.copy2(f, output_drive)
    print(f'Copied {len(glob.glob(os.path.join(output_drive, "*.png")))} results to {output_drive}')
else:
    print('No results directory found.')

No results directory found.


## Troubleshooting

**Model hangs at "Running in eps-prediction mode":**
- It's downloading Paint-by-Example UNet from HuggingFace (~2GB). Wait 2-5 min.
- If it hangs longer than 10 min, check internet connectivity.

**`ModuleNotFoundError: No module named 'pytorch_lightning'`:**
- Re-run the pip install cell. We use `pytorch-lightning==1.9.5` (Python 3.12 compatible).

**OOM errors:**
- Reduce batch size: `--n_samples 1`
- Use fewer DDIM steps: `--ddim_steps 20`

**Dataset errors (folder imgs num != 3):**
- Each folder in `image-wo-bg/` must contain exactly 3 images (front=1, person=2, back=3)
- Same structure for `cloth/`, `cloth-mask/`, `skeletons/`, `inpaint_mask/`
- `warp_feat/` has one image per folder: `{folder_name}.jpg`